# Day 1: Single LLM Call Use Cases for Telecom Security

## Workshop Overview

Welcome to Day 1 of the **AI-Powered Automation in Telecom Security** workshop!

### What You'll Learn Today

Today we focus on **simple, single LLM call use cases** that can dramatically improve your security operations:

1. **Prompt Engineering** - The foundation of effective AI integration
2. **Email Threat Detection** - Classify and analyze suspicious emails
3. **Incident Report Generation** - Transform raw alerts into executive-ready reports
4. **Network Traffic Anomaly Explanation** - Translate technical data to plain language
5. **CDR Fraud Pattern Analysis** - Identify telecom fraud patterns
6. **Log Event Contextualization** - Add intelligence to cryptic log entries
7. **SIM Swap Investigation** - Hands-on exercise

### Framework: Arshai

We'll use the **Arshai** framework - a powerful, developer-first AI framework that gives you complete control:

- ✅ Direct instantiation (no magic)
- ✅ Clean architecture
- ✅ Multi-provider support (OpenAI, Azure, Gemini, OpenRouter)
- ✅ Easy to understand and extend

### Learning Objectives

By the end of today, you will:
- Understand prompt engineering principles
- Create effective security analysis prompts
- Implement single-call LLM solutions
- Calculate ROI for AI automation
- Be ready to implement these in your organization

---

## Setup Instructions

### 1. Install Required Packages

In [1]:
# Install Arshai framework
!pip install arshai -q

print("✅ Arshai framework installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.4 MB/s eta 0:00:00
✅ Arshai framework installed successfully!


### 2. Set Up API Key

We'll use OpenRouter which gives access to multiple models including GPT-4, Claude, and more.

Get your free API key at: https://openrouter.ai/

**For Google Colab users**: You can use Colab's secrets manager to store your API key securely.

In [2]:
import os

# Option 1: Set directly (not recommended for production)
# os.environ["OPENROUTER_API_KEY"] = "your_api_key_here"

# Option 2: For Google Colab - use secrets
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    # Option 3: Load from environment
    if "OPENROUTER_API_KEY" in os.environ:
        print("✅ API key loaded from environment")
    else:
        print("⚠️  Please set OPENROUTER_API_KEY")
        print("   You can set it in the cell above or as an environment variable")

✅ API key loaded from Colab secrets


### 3. Import Required Libraries

In [3]:
import asyncio
import json
from datetime import datetime
from typing import Dict, Any

import logging
logging.basicConfig(level=logging.WARNING)

print("✅ Logging level set to WARNING")

# Arshai imports
from arshai.core.interfaces.illm import ILLMConfig, ILLMInput
from arshai.core.interfaces.iagent import IAgentInput
from arshai.llms.openrouter import OpenRouterClient
from arshai.agents.base import BaseAgent

print("✅ All imports successful!")

✅ Logging level set to WARNING


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'allow_mutation' has been removed
* 'smart_union' has been removed
  warnings.warn(message, UserWarning)


✅ All imports successful!


### 4. Initialize LLM Client

We'll create a reusable LLM client that we'll use throughout the workshop.

In [5]:
# Configure LLM
llm_config = ILLMConfig(
    model="openai/gpt-4o-mini",  # Fast and cost-effective
    temperature=0.0,              # Lower temperature for more consistent security analysis
    max_tokens=2000               # Enough for detailed analysis
)

# Create LLM client
llm_client = OpenRouterClient(llm_config)

print("✅ LLM client initialized")
print(f"   Model: {llm_config.model}")
print(f"   Temperature: {llm_config.temperature}")

2025-10-07 10:37:39.722 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Initializing OpenRouterClient with model: openai/gpt-4o-mini
2025-10-07 10:37:39.724 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - LLM-friendly observability enabled for openrouter
2025-10-07 10:37:39.725 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Creating OpenRouter client with safe HTTP configuration
2025-10-07 10:37:39.790 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - OpenRouter client created successfully with safe configuration
2025-10-07 10:37:39.791 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Closed OpenRouter httpx client
✅ LLM client initialized
   Model: openai/gpt-4o-mini
   Temperature: 0.0


---

## Part 1: Prompt Engineering Fundamentals

### Why Prompt Engineering Matters

The quality of AI output directly depends on the quality of your prompts. A bad prompt gives vague, unusable results. A good prompt gives actionable, consistent intelligence.

### The Framework: Role + Task + Constraints + Output

1. **Role**: Define who the AI is ("senior cybersecurity analyst")
2. **Task**: What exactly should it do ("analyze this email for threats")
3. **Constraints**: Boundaries and requirements ("be concise", "focus on actionable items")
4. **Output**: Specific format expected (JSON, structured report, bullet points)

### Example: Email Threat Analysis

Let's compare bad vs. good prompts using the same email.

In [6]:
# Sample suspicious email
SAMPLE_EMAIL = """
From: it-support@company-secure.com
Subject: Urgent: Verify Your Account

Dear user,

Your account has been flagged for suspicious activity.
Click here to verify: http://bit.ly/verify-acc-now

Failure to verify within 24 hours will result in account suspension.

IT Security Team
"""

print("Sample Email to Analyze:")
print("="*60)
print(SAMPLE_EMAIL)

Sample Email to Analyze:

From: it-support@company-secure.com
Subject: Urgent: Verify Your Account

Dear user,

Your account has been flagged for suspicious activity.
Click here to verify: http://bit.ly/verify-acc-now

Failure to verify within 24 hours will result in account suspension.

IT Security Team



#### ❌ Bad Prompt Example

In [7]:
async def analyze_with_bad_prompt(email: str):
    """Example of a bad prompt - too vague, no structure."""

    bad_prompt = """You are a security expert. Look at this email and tell me if it's dangerous.
      Check for phishing and malware. Be detailed in your response and explain everything you find.
      Make sure to be thorough and don't miss anything important."""

    llm_input = ILLMInput(
        system_prompt=bad_prompt,
        user_message=email
    )

    result = await llm_client.chat(llm_input)
    return result.get('llm_response', '')

# Run the bad prompt
print("\n❌ BAD PROMPT RESULT:")
print("="*60)
bad_result = await analyze_with_bad_prompt(SAMPLE_EMAIL)
print(bad_result)
print("\n⚠️  Problems: Vague, no actionable info, no risk level, inconsistent format")


❌ BAD PROMPT RESULT:
2025-10-07 10:40:06.553 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-07 10:40:07.973 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-07 10:40:07.974 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
Analyzing the email you provided, there are several red flags that suggest it could be a phishing attempt. Here’s a detailed breakdown of the potential dangers:

### 1. **Sender's Email Address**
   - **Email Domain**: The sender's email address is `it-support@company-secure.com`. While it may appear legitimate at first glance, it’s important to verify if this domain is actually associated with your organization. Phishers often use email addresses that look similar to legitimate ones but may have 

#### ✅ Good Prompt Example

In [9]:
async def analyze_with_good_prompt(email: str):
    """Example of a good prompt - structured, specific, actionable."""

    good_prompt = """You are a senior cybersecurity analyst specializing in email threat detection for a telecommunications company.

TASK: Analyze the following email for security threats.

PROVIDE YOUR ANALYSIS IN THIS FORMAT:

1. THREAT VERDICT: [SAFE/SUSPICIOUS/MALICIOUS]
2. CONFIDENCE LEVEL: [percentage]
3. THREAT INDICATORS: [List specific red flags found]
4. ATTACK TYPE: [Phishing/Spear-phishing/Malware/Spam/Legitimate]
5. RISK SCORE: [1-100]
6. POTENTIAL IMPACT: [What could happen if user interacts]
7. RECOMMENDED ACTION: [Specific steps: Quarantine/Block sender/User warning/Allow]
8. REASONING: [Brief explanation of your analysis]

Keep analysis concise and actionable for SOC team.
"""

    llm_input = ILLMInput(
        system_prompt=good_prompt,
        user_message=email
    )

    result = await llm_client.chat(llm_input)
    return result.get('llm_response', '')

# Run the good prompt
print("\n✅ GOOD PROMPT RESULT:")
print("="*60)
good_result = await analyze_with_good_prompt(SAMPLE_EMAIL)
print(good_result)
print("\n✅ Benefits: Structured, actionable, consistent, detailed reasoning")


✅ GOOD PROMPT RESULT:
2025-10-07 10:49:15.174 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-07 10:49:15.818 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-07 10:49:15.820 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
1. THREAT VERDICT: SUSPICIOUS
2. CONFIDENCE LEVEL: 85%
3. THREAT INDICATORS: 
   - Use of a generic greeting ("Dear user") instead of a personalized salutation.
   - Urgency and threat of account suspension to provoke immediate action.
   - Use of a shortened URL (bit.ly) which can obscure the true destination.
   - Email address appears to be from a suspicious domain (company-secure.com) that may not match the legitimate company domain.
4. ATTACK TYPE: Phishing
5. RISK SCORE: 75
6. POTENTIAL IMP

### 🎯 Key Takeaway

**Bad Prompt**: "Is this email suspicious?"
- Vague response
- No actionable information
- Inconsistent format

**Good Prompt**: Structured with Role + Task + Constraints + Output
- Detailed analysis
- Actionable recommendations
- Consistent, parseable format
- Clear reasoning

---

## Part 2: Example 1 - Email Threat Detection Agent

Now let's build a reusable agent that can analyze any email using our good prompt pattern.

In [18]:
class EmailThreatDetectionAgent(BaseAgent):
    """Agent specialized in detecting email threats."""

    def __init__(self, llm_client):
        system_prompt = """You are a senior cybersecurity analyst specializing in email threat detection for a telecommunications company.

TASK: Analyze emails for security threats.

PROVIDE YOUR ANALYSIS IN THIS FORMAT (in Persian):

1. THREAT VERDICT: [SAFE/SUSPICIOUS/MALICIOUS]
2. CONFIDENCE LEVEL: [percentage]
3. THREAT INDICATORS: [List specific red flags found]
4. ATTACK TYPE: [Phishing/Spear-phishing/Malware/Spam/Legitimate]
5. RISK SCORE: [1-100]
6. POTENTIAL IMPACT: [What could happen if user interacts]
7. RECOMMENDED ACTION: [Specific steps]
8. REASONING: [Brief explanation]
9. SENDER_GENDER: [Male/Female/None]

Keep analysis concise and actionable for SOC team."""

        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Process an email and return threat analysis."""

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "analysis": result.get('llm_response', ''),
            "timestamp": datetime.now().isoformat(),
            "model": self.llm_client.config.model
        }

print("✅ EmailThreatDetectionAgent created")

✅ EmailThreatDetectionAgent created


### Test with Multiple Email Examples

In [21]:
# 📧 EXPANDED EMAIL DATASET - 5 Diverse Test Cases
# Demonstrates LLM behavior across difficulty levels and edge cases

EMAIL_DATASET = [
  {
    "id": 1,
    "name": "Classic Phishing - Generic",
    "type": "phishing",
    "difficulty": "easy",
    "content": "From: it-support@company-secure.com\nSubject: Urgent: Verify Your Account\n\nDear user,\n\nYour account has been flagged for suspicious activity.\nClick here to verify: http://bit.ly/verify-acc-now\n\nFailure to verify within 24 hours will result in account suspension.\n\nIT Security Team"
  },
  {
    "id": 2,
    "name": "Legitimate - Internal Communication",
    "type": "legitimate",
    "difficulty": "easy",
    "content": "From: support@company.com\nSubject: Monthly Security Update - October 2025\n\nHello Team,\n\nOur scheduled security maintenance will occur this Saturday, 10 PM - 2 AM GMT.\nNo action required from your side. Systems will be briefly unavailable during this window.\n\nAffected services:\n- Customer portal (down 30 minutes)\n- Internal ticketing system (down 15 minutes)\n\nFor questions, reply to this email or call ext. 5500.\n\nBest regards,\nIT Security Department\nCompany Ltd"
  },
  {
    "id": 3,
    "name": "Spear Phishing - CEO Fraud",
    "type": "spear_phishing",
    "difficulty": "medium",
    "content": "From: ceo@company.com\nSubject: RE: Urgent Wire Transfer Needed\n\nHi,\n\nI'm in a meeting with board members and need you to process an urgent wire transfer immediately.\n\nAmount: $45,000\nBeneficiary: Global Solutions Ltd\nAccount: 8472-3847-2847\nBank: International Trust Bank, Cayman Islands\nSwift: INTLCAYM\n\nPlease handle this immediately and confirm once done.\nDO NOT call me, I'm in meetings all day. Just email confirmation.\n\nThanks,\nJohn Smith\nCEO"
  },
  {
    "id": 4,
    "name": "Sophisticated Phishing - Typosquatting",
    "type": "phishing",
    "difficulty": "hard",
    "content": "From: security-alerts@comp4ny.com\nSubject: Security Alert: New device login detected\n\nHi there,I am miss Mary,\n\nWe detected a new device accessing your account from Russia.\n\nDevice: Windows PC\nLocation: Moscow, Russia\nTime: 2025-10-06 03:42 AM\n\nIf this wasn't you, please secure your account immediately:\nhttps://comp4ny.com/security/verify?token=a8f4d9e2\n\nThis link expires in 2 hours.\n\nBest regards,\nSecurity Team\nCompany Security Operations"
  },
  {
    "id": 5,
    "name": "Legitimate - Vendor Communication",
    "type": "legitimate",
    "difficulty": "medium",
    "content": "From: invoicing@microsoft.com\nSubject: Your Microsoft Azure Invoice - October 2025\n\nDear Customer,\n\nYour monthly Azure invoice is now available.\n\nAccount: company-azure-prod\nInvoice ID: INV-2025-10-847521\nAmount: $1,247.50\nDue Date: October 25, 2025\n\nView invoice: https://portal.azure.com/invoices/INV-2025-10-847521\n\nPayment methods: Existing credit card ending in 4892\n\nFor support, visit: https://azure.microsoft.com/support\n\nThank you for using Microsoft Azure.\n\nMicrosoft Corporation\nOne Microsoft Way, Redmond, WA 98052"
  }
]

print(f"✅ Created dataset with {len(EMAIL_DATASET)} test emails")
print("\n📋 Test Cases:")
for email in EMAIL_DATASET:
    print(f"   {email['id']}. {email['name']} ({email['type']}, {email['difficulty']})")

✅ Created dataset with 5 test emails

📋 Test Cases:
   1. Classic Phishing - Generic (phishing, easy)
   2. Legitimate - Internal Communication (legitimate, easy)
   3. Spear Phishing - CEO Fraud (spear_phishing, medium)
   4. Sophisticated Phishing - Typosquatting (phishing, hard)
   5. Legitimate - Vendor Communication (legitimate, medium)


In [22]:
# Create the agent
email_agent = EmailThreatDetectionAgent(llm_client)

# Analyze each email
print("\n" + "="*80)
print("EMAIL THREAT DETECTION ANALYSIS")
print("="*80)

for email in EMAIL_DATASET:
    print(f"\n📧 Email #{email['id']} (Expected: {email['type']})")
    print("-" * 80)

    # Analyze
    agent_input = IAgentInput(message=email['content'])
    result = await email_agent.process(agent_input)

    print(result['analysis'])
    print(f"\n⏱️  Analyzed at: {result['timestamp']}")

print("\n" + "="*80)
print("✅ Analysis complete!")


EMAIL THREAT DETECTION ANALYSIS

📧 Email #1 (Expected: phishing)
--------------------------------------------------------------------------------
2025-10-07 11:15:16.814 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-07 11:15:17.545 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-07 11:15:17.546 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
1. THREAT VERDICT: SUSPICIOUS  
2. CONFIDENCE LEVEL: 75%  
3. THREAT INDICATORS:  
   - Urgent language ("Failure to verify within 24 hours")  
   - Use of a shortened URL (bit.ly)  
   - Generic greeting ("Dear user")  
   - Request for immediate action  
4. ATTACK TYPE: Phishing  
5. RISK SCORE: 70  
6. POTENTIAL IMPACT: User credentials could be compromised, leading to un

### 💰 ROI Calculation

**Manual Process:**
- Average time per email: 3-5 minutes
- Daily emails to review: 1,000+
- Required analysts: 8-10 people working full-time

**AI-Powered Process:**
- Average time per email: 3-5 seconds
- Daily capacity: Virtually unlimited
- Required analysts: 1-2 for escalation review

**Savings:**
- Time savings: 99%+
- Cost savings: 80-90%
- Consistency: 100% (no fatigue-based errors)

---

## Part 3: Example 2 - Security Incident Report Generation

Convert raw technical alerts into executive-readable incident reports.

In [15]:
class IncidentReportAgent(BaseAgent):
    """Agent that generates executive-friendly incident reports."""

    def __init__(self, llm_client):
        system_prompt = """You are a senior SOC analyst who writes incident reports for executive leadership.

TASK: Convert technical security alerts into clear, executive-readable incident summaries.

STRUCTURE YOUR REPORT AS FOLLOWS (in Persian):

INCIDENT SUMMARY REPORT
Generated: [timestamp]
Priority: [LOW/MEDIUM/HIGH/CRITICAL]

WHAT HAPPENED:
[1-2 sentences in plain language]

WHY IT MATTERS:
[3-4 bullet points explaining business impact]

IMMEDIATE RISK:
[What could happen if not addressed]

RECOMMENDED ACTIONS:
[Numbered list of specific steps, prioritized]

ESTIMATED RESPONSE TIME: [time estimate]
BUSINESS IMPACT IF UNADDRESSED: [risk level]

Use plain language. Avoid jargon. Focus on business impact and actionable steps."""

        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Generate incident report from raw alert data."""

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "report": result.get('llm_response', ''),
            "generated_at": datetime.now().isoformat()
        }

print("✅ IncidentReportAgent created")

✅ IncidentReportAgent created


In [23]:
# 🚨 EXPANDED INCIDENT ALERTS - 5 Diverse Scenarios
# Shows different severity levels and attack types

RAW_ALERTS = [
  {
    "alert_id": "AUTH-47821",
    "name": "Brute Force Attack from Foreign IP",
    "severity": "high",
    "raw_data": "[2025-10-06 14:23:41] AUTHENTICATION ALERT\nEvent: Multiple failed login attempts\nUser: admin\nSource IP: 185.220.101.47\nLocation: Russia\nPrevious successful login: UK (2 hours ago)\nFailed attempts: 15 in last 5 minutes\nAccount status: Active\nUser role: System Administrator\nMFA Status: Not completed\nUnusual login hours: 2:23 AM local time"
  },
  {
    "alert_id": "DB-59284",
    "name": "Suspicious Database Query Pattern",
    "severity": "critical",
    "raw_data": "[2025-10-06 11:34:18] DATABASE ALERT\nEvent: Unusual query pattern detected\nUser: db_backup_service\nQuery: SELECT * FROM customer_credentials WHERE status='active'\nRows returned: 45,891\nQuery time: 0.34s\nSource IP: 10.30.15.47\nNormal query pattern: SELECT name, account_id FROM customer_credentials WHERE last_modified > DATE\nTypical rows returned: 200-500 (incremental changes only)\nDeviation: Query retrieved ALL columns including passwords, security_questions, 2fa_secrets\nLast normal backup: 6 hours ago"
  },
  {
    "alert_id": "NET-84729",
    "name": "Unusual Outbound Data Transfer",
    "severity": "medium",
    "raw_data": "[2025-10-06 16:45:33] NETWORK ALERT\nEvent: Large outbound data transfer\nSource: Internal server 10.20.15.84 (File server)\nDestination: 89.248.172.16 (External IP, Netherlands)\nProtocol: HTTPS\nData transferred: 4.7 GB\nDuration: 23 minutes\nNormal behavior: This server typically has <100MB outbound daily\nFiles transferred: Multiple .zip archives\nDestination reputation: Unknown (not in threat databases)\nUser session: jdoe (Junior Marketing Coordinator)"
  },
  {
    "alert_id": "MALWARE-19384",
    "name": "Malware Detection on Endpoint",
    "severity": "high",
    "raw_data": "[2025-10-06 09:12:47] ENDPOINT PROTECTION ALERT\nEvent: Malware detected and quarantined\nHostname: DESKTOP-SALES-47\nUser: sthompson (Sales Manager)\nMalware type: Trojan.Downloader.Generic\nFile: invoice_Q3_2025_final.exe\nLocation: C:\\Users\\sthompson\\Downloads\\\nDetection method: Behavioral analysis\nActions taken: File quarantined, process terminated\nOrigin: Email attachment from external sender (vendors@supplierltd.com)\nRisk: High - Trojan attempted to establish C2 connection\nC2 Domain blocked: mal-server-2891.biz"
  },
  {
    "alert_id": "ACCESS-62847",
    "name": "After-Hours Access to Sensitive System",
    "severity": "medium",
    "raw_data": "[2025-10-06 03:28:15] ACCESS CONTROL ALERT\nEvent: Sensitive system accessed during off-hours\nUser: mjohnson (Database Administrator)\nSystem: Production customer database\nAccess time: 3:28 AM (Outside normal hours: 8 AM - 6 PM)\nLocation: Home IP (VPN connection)\nActions performed:\n  - Exported customer contact list (2,847 records)\n  - Modified 3 customer records\n  - Downloaded database schema\nDuration: 47 minutes\nHistorical pattern: User rarely accesses system after 7 PM\nLast after-hours access: 6 months ago (documented incident response)"
  }
]

print(f"✅ Created dataset with {len(RAW_ALERTS)} incident alerts")
print("\n📋 Alert Types:")
for alert in RAW_ALERTS:
    print(f"   {alert['alert_id']}: {alert['name']} ({alert['severity']})")

✅ Created dataset with 5 incident alerts

📋 Alert Types:
   AUTH-47821: Brute Force Attack from Foreign IP (high)
   DB-59284: Suspicious Database Query Pattern (critical)
   NET-84729: Unusual Outbound Data Transfer (medium)
   MALWARE-19384: Malware Detection on Endpoint (high)
   ACCESS-62847: After-Hours Access to Sensitive System (medium)


In [24]:
# Create the agent for Incident Report Generation
incident_report_agent = IncidentReportAgent(llm_client)

# Analyze each raw alert and generate a report
print("\n" + "="*80)
print("INCIDENT REPORT GENERATION")
print("="*80)

for alert in RAW_ALERTS:
    print(f"\n🚨 Alert ID: {alert['alert_id']} (Severity: {alert['severity']})")
    print("-" * 80)

    # Analyze
    agent_input = IAgentInput(message=alert['raw_data'])
    result = await incident_report_agent.process(agent_input)

    print(result['report'])
    print(f"\n⏱️  Generated at: {result['generated_at']}")
    print("-" * 80)

print("\n" + "="*80)
print("✅ Report generation complete!")


INCIDENT REPORT GENERATION

🚨 Alert ID: AUTH-47821 (Severity: high)
--------------------------------------------------------------------------------
2025-10-07 11:16:35.761 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-07 11:16:36.360 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-07 11:16:36.361 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
گزارش خلاصه حادثه  
تاریخ تولید: 2025-10-06 14:23:41  
اولویت: بالا  

چه اتفاقی افتاد:  
تعداد زیادی تلاش ناموفق برای ورود به حساب کاربری مدیر سیستم با نام کاربری "admin" از یک آدرس IP در روسیه ثبت شده است. این تلاش‌ها در ساعت 2:23 بامداد به وقت محلی انجام شده و 15 بار در 5 دقیقه اخیر صورت گرفته است.

چرا این موضوع اهمیت دارد:  
- احتمال دسترسی غیرمجاز به سیستم‌های حساس 

### 💰 ROI Calculation

**Manual Report Generation:**
- Time per report: 10-15 minutes
- Requires senior analyst
- Delays incident response

**AI-Powered Generation:**
- Time per report: 5 seconds
- Consistent quality
- Immediate availability
- Frees senior analysts for actual response

---

## Part 4: Example 3 - Network Traffic Anomaly Explanation

Translate complex network data into plain language for non-technical stakeholders.

In [ ]:
class NetworkAnomalyExplainerAgent(BaseAgent):
    """Agent that explains network anomalies in plain language."""

    def __init__(self, llm_client):
        system_prompt = """You are a network security expert who explains technical issues to non-technical stakeholders.

TASK: Translate network security alerts into plain language explanations.

PROVIDE YOUR EXPLANATION IN THIS FORMAT (in Persian):

PLAIN LANGUAGE EXPLANATION:
[Explain what's happening using simple analogies. No jargon.]

WHAT'S ABNORMAL:
[Bullet list of unusual behaviors]

BUSINESS IMPACT:
[What this means for the business and customers]

TECHNICAL SUMMARY:
[Brief technical description for IT team]

RECOMMENDED ACTIONS:
[Numbered, prioritized list]

URGENCY: [LOW/MEDIUM/HIGH/CRITICAL]

Use analogies and simple language. Make it understandable for non-technical management."""

        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Explain network anomaly."""

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "explanation": result.get('llm_response', ''),
            "timestamp": datetime.now().isoformat()
        }

print("✅ NetworkAnomalyExplainerAgent created")

✅ NetworkAnomalyExplainerAgent created


In [ ]:
# 🌐 EXPANDED NETWORK ANOMALY ALERTS - 5 Different Complexities
# From simple (streaming) to complex (DNS tunneling)

NETWORK_ALERTS = [
  {
    "alert_id": "NET-47821",
    "name": "DDoS Attack - SIP Flood",
    "complexity": "medium",
    "alert": "Alert ID: NET-47821\nTimestamp: 2025-10-06 09:15:33\nSource: Network Monitoring System\n\nTECHNICAL DATA:\n- Port 5060 traffic spike: 400% above baseline\n- Source IPs: 847 unique addresses (127 countries)\n- Packet pattern: Short duration SIP INVITE requests\n- Target: PBX server cluster (10.50.20.0/24)\n- Duration: Ongoing (23 minutes)\n- Failed authentication rate: 99.7%\n- Legitimate calls impacted: ~150 customers reporting issues\n- Attack classification: VoIP DDoS (SIP flood)"
  },
  {
    "alert_id": "NET-85492",
    "name": "DNS Tunneling Detected",
    "complexity": "hard",
    "alert": "Alert ID: NET-85492\nTimestamp: 2025-10-06 14:32:18\nSource: DNS Traffic Analysis System\n\nTECHNICAL DATA:\n- Unusual DNS query pattern detected\n- Source: Internal workstation 10.30.25.147\n- Queries per minute: 847 (normal: 5-10)\n- Domain pattern: [random].suspicious-domain.xyz\n- Query types: TXT records (unusual for workstation)\n- Payload size: Average 180 bytes per query\n- Response size: Average 220 bytes per response\n- Total data transferred: ~2.4 MB over 15 minutes\n- Pattern matches: Known DNS tunneling signatures\n- Destination DNS server: 8.8.8.8 (Google Public DNS)"
  },
  {
    "alert_id": "NET-29384",
    "name": "Unusual Internal Port Scanning",
    "complexity": "medium",
    "alert": "Alert ID: NET-29384\nTimestamp: 2025-10-06 11:47:22\nSource: Intrusion Detection System\n\nTECHNICAL DATA:\n- Activity: Systematic port scanning\n- Source: 10.40.15.92 (Employee workstation)\n- Target: Internal network 10.50.0.0/16 (Data center subnet)\n- Ports scanned: 80, 443, 22, 3389, 3306, 1433, 5432, 27017\n- Scan rate: 50 ports per second\n- Unique targets: 254 IP addresses (full subnet scan)\n- Duration: 18 minutes\n- User context: kbrown (Finance Department, Accountant)\n- Normal behavior: User typically accesses 3-4 specific servers only"
  },
  {
    "alert_id": "NET-74821",
    "name": "Bandwidth Spike - Video Streaming",
    "complexity": "low",
    "alert": "Alert ID: NET-74821\nTimestamp: 2025-10-06 13:15:47\nSource: Bandwidth Monitoring\n\nTECHNICAL DATA:\n- Event: Significant bandwidth usage spike\n- Source: Conference room network (10.60.10.0/24)\n- Destinations: Multiple video streaming services (Netflix, YouTube, Twitch)\n- Bandwidth consumed: 450 Mbps (75% of allocated conference room bandwidth)\n- Duration: 2 hours (ongoing)\n- Concurrent streams: 3\n- Business impact: Video conference quality degraded in adjacent rooms\n- Time: Business hours (1:15 PM)\n- Building occupancy: 87% (normal for afternoon)"
  },
  {
    "alert_id": "NET-38492",
    "name": "Cryptocurrency Mining Traffic",
    "complexity": "medium",
    "alert": "Alert ID: NET-38492\nTimestamp: 2025-10-06 22:47:15\nSource: Network Traffic Analysis\n\nTECHNICAL DATA:\n- Activity: Suspected cryptocurrency mining\n- Source: Server VM-WEBTEST-04 (Development environment)\n- Destinations: Known mining pools (cryptopool.io, minepool.net)\n- Port: 3333 (Stratum mining protocol)\n- Traffic pattern: Persistent outbound connections\n- CPU utilization on source: 95% (normal: 10-15%)\n- Network traffic: 15 Mbps sustained\n- Duration: 6 hours (detected at 10:47 PM, started ~4:45 PM)\n- VM owner: Development team (shared resource)\n- Business impact: Resource consumption affecting test environment performance"
  }
]

print(f"✅ Created dataset with {len(NETWORK_ALERTS)} network alerts")
print("\n📋 Alert Complexities:")
for alert in NETWORK_ALERTS:
    print(f"   {alert['alert_id']}: {alert['name']} ({alert['complexity']})")

✅ Created dataset with 5 network alerts

📋 Alert Complexities:
   NET-47821: DDoS Attack - SIP Flood (medium)
   NET-85492: DNS Tunneling Detected (hard)
   NET-29384: Unusual Internal Port Scanning (medium)
   NET-74821: Bandwidth Spike - Video Streaming (low)
   NET-38492: Cryptocurrency Mining Traffic (medium)


In [ ]:
# Create the agent for Network Anomaly Explanation
network_explainer_agent = NetworkAnomalyExplainerAgent(llm_client)

# Explain each network alert
print("\n" + "="*80)
print("NETWORK ANOMALY EXPLANATION")
print("="*80)

for alert in NETWORK_ALERTS:
    print(f"\n🌐 Alert ID: {alert['alert_id']} ({alert['name']})")
    print("-" * 80)

    # Explain
    agent_input = IAgentInput(message=alert['alert'])
    result = await network_explainer_agent.process(agent_input)

    print(result['explanation'])
    print(f"\n⏱️  Explained at: {result['timestamp']}")
    print("-" * 80)

print("\n" + "="*80)
print("✅ Explanation complete!")


NETWORK ANOMALY EXPLANATION

🌐 Alert ID: NET-47821 (DDoS Attack - SIP Flood)
--------------------------------------------------------------------------------
2025-10-06 19:37:40.819 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-06 19:37:42.019 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-06 19:37:42.020 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
PLAIN LANGUAGE EXPLANATION:
تصور کنید که یک رستوران محبوب در یک روز شلوغ، ناگهان با تعداد زیادی مشتری که فقط می‌خواهند از منو سوال کنند، مواجه می‌شود. این مشتریان به جای سفارش غذا، فقط سوالات بی‌پاسخ می‌پرسند و در نتیجه، مشتریان واقعی که می‌خواهند غذا سفارش دهند، نمی‌توانند به خدمات دسترسی پیدا کنند. در اینجا، سرور تلفن ما (PBX) مانند رستوران است و این حمله به نو

---

## Part 5: Example 4 - CDR Fraud Pattern Analysis

Analyze Call Detail Records (CDR) to identify potential telecom fraud.

In [ ]:
class CDRFraudAnalystAgent(BaseAgent):
    """Agent specialized in detecting telecom fraud from CDR data."""

    def __init__(self, llm_client):
        system_prompt = """You are an expert telecom fraud analyst with 15 years of experience.

TASK: Analyze Call Detail Records (CDR) to identify potential fraud patterns.

PROVIDE YOUR ANALYSIS IN THIS FORMAT (in Persian):

FRAUD ASSESSMENT REPORT

VERDICT: [LIKELY FRAUD / SUSPICIOUS / NORMAL] (Confidence: XX%)

FRAUD TYPE: [IRSF/Wangiri/SIM Box/PBX Hacking/Subscription Fraud/Legitimate]

RED FLAGS IDENTIFIED:
[Numbered list of specific suspicious indicators]

FRAUD MECHANICS:
[Explain HOW the fraud is being executed]

FINANCIAL IMPACT:
- Current loss: [estimated]
- Projected loss if unaddressed: [24-hour estimate]

RECOMMENDED ACTIONS (Immediate):
[Specific, numbered steps]

RECOMMENDED ACTIONS (Strategic):
[Prevention measures]

URGENCY: [LOW/MEDIUM/HIGH/CRITICAL]

Focus on International Revenue Share Fraud (IRSF), Wangiri, and unusual calling patterns."""

        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Analyze CDR for fraud patterns."""

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "analysis": result.get('llm_response', ''),
            "analyzed_at": datetime.now().isoformat()
        }

print("✅ CDRFraudAnalystAgent created")

✅ CDRFraudAnalystAgent created


In [ ]:
# 📞 EXPANDED CDR FRAUD DATASET - 5 Diverse Cases
# Mix of fraud types and legitimate calls

CDR_SAMPLES = [
  {
    "cdr_id": "CDR-19847521",
    "name": "IRSF - Premium Rate Fraud",
    "fraud_type": "irsf",
    "expected_verdict": "fraud",
    "data": "CDR-ID: 19847521\nTimestamp: 2025-10-06 03:47:12\nAccount: +44-7700-900123\nCall type: International\nDestination: +881-6 (Satellite premium service)\nDuration: 47 minutes\nCost: \u00a394.50\nAccount age: 8 days\nPrevious international calls: 0\nCall initiated: 3:47 AM (unusual time)\nSimilar pattern calls: 23 in last hour (same account)\nTotal cost last hour: \u00a32,173.50\nCustomer notification attempts: 2 (no response)\nSIM card: Recently replaced (3 days ago)\nLocation: Tower indicates London, but roaming shows as \"unregistered\"\n"
  },
  {
    "cdr_id": "CDR-19847845",
    "name": "Legitimate Business Call - USA",
    "fraud_type": "none",
    "expected_verdict": "legitimate",
    "data": "CDR-ID: 19847845\nTimestamp: 2025-10-06 14:22:08\nAccount: +44-7700-900456\nCall type: International\nDestination: +1-212-555-0147 (USA - New York landline)\nDuration: 12 minutes\nCost: \u00a32.40\nAccount age: 847 days\nPrevious international calls: 45 (last 90 days)\nCall initiated: 2:22 PM (business hours)\nCustomer profile: Business account, frequently calls US and EU\nAverage monthly spend: \u00a3450\nCustomer contact history: Registered business name \"TechCorp UK Ltd\"\nDestination number: Verified business (Google Inc. office)\nCall pattern: Regular, similar calls 2-3 times per week\n"
  },
  {
    "cdr_id": "CDR-28374819",
    "name": "Wangiri Fraud Attempt",
    "fraud_type": "wangiri",
    "expected_verdict": "fraud",
    "data": "CDR-ID: 28374819\nTimestamp: 2025-10-06 16:33:21\nAccount: +44-7700-900789\nCall type: Incoming international\nSource: +225-8472-847-392 (Ivory Coast)\nDuration: 3 seconds (one ring)\nCall pattern: Missed call\nAccount age: 234 days\nSimilar missed calls: 847 customers received identical pattern\nSource number pattern: Sequential (+225-8472-847-391 to +225-8472-847-450)\nKnown fraud database: Number flagged in IRSF database\nCallback attempts: 23 customers already called back\nCallback costs: Premium rate (\u00a34.50/minute)\nTotal estimated fraud if unchecked: \u00a347,000 (based on callback rate)\n"
  },
  {
    "cdr_id": "CDR-48372910",
    "name": "PBX Hacking - After Hours",
    "fraud_type": "pbx_hacking",
    "expected_verdict": "fraud",
    "data": "CDR-ID: 48372910\nTimestamp: 2025-10-06 02:15:47\nAccount: +44-20-7946-0958 (Business PBX line)\nCall type: Outbound international\nDestination: Multiple premium destinations (Somalia, Estonia premium, Satellite)\nCall duration: 2 hours 47 minutes continuous\nCost: \u00a32,847.50\nCalls originated: 47 simultaneous calls\nAccount type: Business PBX with 20 extensions\nCall initiated: 2:15 AM (business closed 8 PM - 8 AM)\nNormal pattern: 0 calls after 7 PM\nAuthentication: Auto-attendant IVR system was accessed\nAccess method: Sequential extension dialing (801, 802, 803... suggesting brute force)\nCaller ID spoofing detected: Outbound calls showed customer's main number\n"
  },
  {
    "cdr_id": "CDR-83749201",
    "name": "Legitimate - Family Call to Home Country",
    "fraud_type": "none",
    "expected_verdict": "legitimate",
    "data": "CDR-ID: 83749201\nTimestamp: 2025-10-06 19:45:33\nAccount: +44-7700-900234\nCall type: International\nDestination: +92-321-555-8472 (Pakistan mobile)\nDuration: 1 hour 23 minutes\nCost: \u00a312.75\nAccount age: 1,847 days (5+ years)\nPrevious international calls: 847 calls to same country (last 12 months)\nCall initiated: 7:45 PM (evening, typical for personal calls)\nCall pattern: Every Sunday evening at similar time\nDestination: Same number called 156 times in past year\nCustomer profile: Retail customer, good payment history\nPayment status: All bills paid on time, no payment issues\nAverage monthly spend: \u00a385 (including international calls)\nCustomer verification: Previously verified for international calling\n"
  }
]

print(f"✅ Created dataset with {len(CDR_SAMPLES)} CDR records")
print("\n📋 CDR Cases:")
for cdr in CDR_SAMPLES:
    print(f"   {cdr['cdr_id']}: {cdr['name']} ({cdr['fraud_type']})")

✅ Created dataset with 5 CDR records

📋 CDR Cases:
   CDR-19847521: IRSF - Premium Rate Fraud (irsf)
   CDR-19847845: Legitimate Business Call - USA (none)
   CDR-28374819: Wangiri Fraud Attempt (wangiri)
   CDR-48372910: PBX Hacking - After Hours (pbx_hacking)
   CDR-83749201: Legitimate - Family Call to Home Country (none)


In [ ]:
# Create the agent for CDR Fraud Analysis
cdr_fraud_agent = CDRFraudAnalystAgent(llm_client)

# Analyze each CDR sample
print("\n" + "="*80)
print("CDR FRAUD PATTERN ANALYSIS")
print("="*80)

for cdr in CDR_SAMPLES:
    print(f"\n📞 CDR ID: {cdr['cdr_id']} ({cdr['name']})")
    print("-" * 80)

    # Analyze
    agent_input = IAgentInput(message=cdr['data'])
    result = await cdr_fraud_agent.process(agent_input)

    print(result['analysis'])
    print(f"\n⏱️  Analyzed at: {result['analyzed_at']}")
    print("-" * 80)

print("\n" + "="*80)
print("✅ Analysis complete!")


CDR FRAUD PATTERN ANALYSIS

📞 CDR ID: CDR-19847521 (IRSF - Premium Rate Fraud)
--------------------------------------------------------------------------------
2025-10-06 19:39:34.054 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-06 19:39:34.921 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-06 19:39:34.922 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
گزارش ارزیابی تقلب

حکم: مشکوک (اعتماد: 85%)

نوع تقلب: IRSF

پرچم‌های قرمز شناسایی شده:
1. تماس بین‌المللی به مقصد شماره‌ای با پیش‌شماره +881-6 (خدمات پریمیوم ماهواره‌ای).
2. مدت زمان تماس 47 دقیقه که هزینه بالایی را به همراه دارد (£94.50).
3. سن حساب تنها 8 روز است و هیچ تماس بین‌المللی قبلی ندارد.
4. زمان تماس در ساعت 3:47 صبح که زمان غیرمعمولی است.
5. 23 تم

### 💰 ROI Calculation for CDR Fraud Detection

**Impact of Telecom Fraud:**
- Global telecom fraud losses: $38.95 billion annually (CFCA)
- IRSF average loss per incident: £15,000
- Detection time matters: Hours of delay = thousands in losses

**Manual CDR Analysis:**
- Time per CDR: 2-5 minutes
- Daily CDRs to review: 10,000+
- Most fraud detected after damage done

**AI-Powered Analysis:**
- Time per CDR: 3-5 seconds
- Can analyze 100% of CDRs in real-time
- Catches fraud in minutes, not hours

---

## Part 6: Example 5 - Log Event Contextualization

Turn cryptic log entries into actionable security intelligence.

In [ ]:
class LogContextualizationAgent(BaseAgent):
    """Agent that adds context and intelligence to log events."""

    def __init__(self, llm_client):
        system_prompt = """You are a database security analyst expert at identifying suspicious patterns.

TASK: Analyze log entries and provide security context.

PROVIDE YOUR ANALYSIS IN THIS FORMAT (in Persian):

SUSPICIOUS ACTIVITY ANALYSIS

WHAT HAPPENED:
[Plain language description]

WHY THIS IS SUSPICIOUS:
[Numbered list of red flags]

NORMAL vs. OBSERVED BEHAVIOR:
- Normal: [expected behavior]
- Observed: [what was actually seen]

POTENTIAL THREAT SCENARIOS:
[List possible explanations, ranked by likelihood]

SEVERITY: [LOW/MEDIUM/HIGH/CRITICAL]

RECOMMENDED IMMEDIATE ACTIONS:
[Numbered, time-sensitive steps]

RECOMMENDED INVESTIGATION:
[What to check next]

TIME TO RESPOND: [urgency estimate]

Focus on data exfiltration, privilege escalation, and insider threats."""

        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Contextualize log entry."""

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=input.message
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "contextualization": result.get('llm_response', ''),
            "timestamp": datetime.now().isoformat()
        }

print("✅ LogContextualizationAgent created")

✅ LogContextualizationAgent created


In [ ]:
# 📝 EXPANDED LOG ENTRIES - 5 Different Scenarios
# Critical threats, legitimate activity, and user errors

LOG_ENTRIES = [
  {
    "log_id": "LOG-28471",
    "name": "Data Exfiltration - Suspicious Query",
    "severity": "critical",
    "entry": "[2025-10-06 11:34:18] DATABASE_LOG\nEvent: Unusual database query\nUser: db_backup_service\nQuery: SELECT * FROM customer_credentials WHERE status='active'\nRows returned: 45,891\nQuery time: 0.34s\nSource IP: 10.30.15.47\n\nHistorical context:\nUsual query pattern: SELECT name, account_id FROM customer_credentials WHERE last_modified > DATE\nTypical rows returned: 200-500 (incremental changes only)\nTypical query time: 0.05s\nColumns normally accessed: name, account_id, last_modified\nColumns accessed this time: ALL (including passwords, security_questions, 2fa_secrets, payment_methods)\nUser account created: 847 days ago (normal service account)\nLast password change: Today, 11:00 AM (15 minutes before this query)\n"
  },
  {
    "log_id": "LOG-83729",
    "name": "Privilege Escalation Attempt",
    "severity": "high",
    "entry": "[2025-10-06 15:22:41] SYSTEM_LOG\nEvent: Unauthorized privilege escalation attempt\nUser: mwilson (Standard user)\nAction: Attempted to execute 'sudo su -' command\nResult: DENIED (user not in sudoers file)\nSource: Terminal session from 10.40.25.18\nUser role: Junior Developer\nNormal permissions: Read-only access to development servers\n\nAdditional context:\nFailed attempts in sequence:\n  15:22:38 - sudo su - (DENIED)\n  15:22:41 - sudo su - (DENIED)\n  15:22:44 - su root (DENIED - incorrect password)\n  15:22:47 - su root (DENIED - incorrect password)\n  15:22:50 - su root (DENIED - incorrect password, account locked)\n\nUser behavior history:\n  - Never attempted elevated privileges before\n  - Account created 89 days ago\n  - Recently accessed: Sensitive repository containing infrastructure code\n  - Unusual login pattern: Logged in at 3:15 AM today (never done before)\n"
  },
  {
    "log_id": "LOG-19283",
    "name": "Legitimate System Maintenance",
    "severity": "low",
    "entry": "[2025-10-06 02:00:15] SYSTEM_LOG\nEvent: Automated system maintenance started\nScript: weekly_cleanup.sh\nUser: sys_maintenance (service account)\nActions performed:\n  - Cleared temporary files: 4.2 GB freed\n  - Rotated log files: 15 files archived\n  - Updated package indexes\n  - Checked disk space: All partitions <75%\n  - Ran database optimization queries\n  - Backed up configuration files\nDuration: 23 minutes\nResult: SUCCESS (all tasks completed)\nNext scheduled run: 2025-10-13 02:00:00\n\nHistorical context:\nThis is a routine weekly maintenance job:\n  - Scheduled every Sunday at 2:00 AM\n  - Has run successfully for 2.5 years (130 consecutive successful runs)\n  - No changes to script in last 6 months\n  - Approved maintenance window: Sundays 2:00 AM - 4:00 AM\n  - Monitoring: Automated alerts if duration exceeds 30 minutes or if failures occur\n"
  },
  {
    "log_id": "LOG-47821",
    "name": "Insider Threat - File Access Pattern",
    "severity": "critical",
    "entry": "[2025-10-06 16:45:29] FILE_ACCESS_LOG\nEvent: Unusual file access pattern\nUser: kbrown (Finance Department)\nFiles accessed: 847 files in 15 minutes\nFile locations:\n  - /finance/confidential/salaries_2025.xlsx (accessed)\n  - /finance/confidential/executive_compensation.xlsx (accessed)\n  - /hr/personnel/employee_records/ (directory listing)\n  - /hr/personnel/termination_plans_q4.docx (accessed)\n  - /legal/contracts/acquisition_target_valuation.pdf (accessed)\n  - /finance/budgets/cost_cutting_proposals.xlsx (accessed)\n\nActions taken:\n  - Copied all files to USB drive (4.7 GB total)\n  - Uploaded copies to personal Dropbox account\n  - Deleted local browser history\n  - Cleared recent files list\n\nUser context:\n  - Role: Junior Financial Analyst\n  - Normal access: Limited to general finance reports only\n  - Permissions: Should NOT have access to executive compensation or HR files\n  - Recent activity:\n      \u2022 Submitted resignation letter yesterday\n      \u2022 Final day: Next Friday\n      \u2022 Recently interviewed at competitor (LinkedIn activity shows)\n  - Access method: Used manager's credentials (password sharing suspected)\n"
  },
  {
    "log_id": "LOG-92847",
    "name": "Failed Login - Legitimate User Error",
    "severity": "low",
    "entry": "[2025-10-06 09:05:22] AUTHENTICATION_LOG\nEvent: Multiple failed login attempts\nUser: jthomas (Marketing Manager)\nFailed attempts: 5 (within 2 minutes)\nSource IP: 10.20.15.84 (Office network)\nDevice: LAPTOP-MARKETING-15 (registered company device)\nTime: 9:05 AM (normal work hours)\nError reason: \"Invalid password\"\nResult: Account temporarily locked for 15 minutes (security policy)\n\nUser context:\n  - Account age: 1,247 days (3.4 years)\n  - Normal login pattern: Daily, 9 AM - 5 PM\n  - Recent password change: Required by policy 2 days ago (90-day rotation)\n  - Previous failed logins: 0 in last 6 months\n\nSubsequent events:\n  09:15 - User contacted IT helpdesk\n  09:18 - Identity verified by helpdesk (security questions passed)\n  09:20 - Account unlocked by IT\n  09:22 - Successful login with correct password\n  09:23 - User reported: \"Forgot I changed password, was using old one\"\n\nRisk assessment: Low - Legitimate user password confusion after forced rotation\n"
  }
]

print(f"✅ Created dataset with {len(LOG_ENTRIES)} log entries")
print("\n📋 Log Cases:")
for log in LOG_ENTRIES:
    print(f"   {log['log_id']}: {log['name']} ({log['severity']})")

✅ Created dataset with 5 log entries

📋 Log Cases:
   LOG-28471: Data Exfiltration - Suspicious Query (critical)
   LOG-83729: Privilege Escalation Attempt (high)
   LOG-19283: Legitimate System Maintenance (low)
   LOG-47821: Insider Threat - File Access Pattern (critical)
   LOG-92847: Failed Login - Legitimate User Error (low)


In [ ]:
# Create the agent for Log Event Contextualization
log_context_agent = LogContextualizationAgent(llm_client)

# Contextualize each log entry
print("\n" + "="*80)
print("LOG EVENT CONTEXTUALIZATION")
print("="*80)

for log in LOG_ENTRIES:
    print(f"\n📝 Log ID: {log['log_id']} ({log['name']})")
    print("-" * 80)

    # Contextualize
    agent_input = IAgentInput(message=log['entry'])
    result = await log_context_agent.process(agent_input)

    print(result['contextualization'])
    print(f"\n⏱️  Contextualized at: {result['timestamp']}")
    print("-" * 80)

print("\n" + "="*80)
print("✅ Contextualization complete!")


LOG EVENT CONTEXTUALIZATION

📝 Log ID: LOG-28471 (Data Exfiltration - Suspicious Query)
--------------------------------------------------------------------------------
2025-10-06 19:40:36.134 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-06 19:40:36.668 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-06 19:40:36.669 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
تحلیل فعالیت مشکوک

WHAT HAPPENED:
یک کاربر با نام "db_backup_service" یک پرس و جوی غیرمعمول به پایگاه داده ارسال کرده است که شامل انتخاب تمام اطلاعات از جدول "customer_credentials" با وضعیت "فعال" می‌باشد. این پرس و جو ۴۵,۸۹۱ ردیف را برگردانده و زمان اجرای آن ۰.۳۴ ثانیه بوده است.

WHY THIS IS SUSPICIOUS:
1. پرس و جوی غیرمعمول: کاربر به جای پرس و جوهای

---

## Part 7: Hands-On Exercise - SIM Swap Investigation Assistant

### The Challenge

Your fraud system flags 1,500 SIM swap requests daily for manual review (risk scores 21-79). Each investigation takes 5-10 minutes. Your team can only review ~200/day, meaning **1,300 slip through uninvestigated**.

### Your Task

Create an AI agent that investigates each flagged request and produces an investigation report for human reviewers.

### Instructions

1. Review the provided SIM swap data structure
2. Design your prompt following the **Role + Task + Constraints + Output** framework
3. Implement the agent
4. Test with sample data
5. Calculate ROI

### Step 1: Review the Data Structure

In [ ]:
# Sample SIM Swap Request Data
SIM_SWAP_REQUEST = {
    "request_id": "SIM-2025-10-06-48271",
    "timestamp": "2025-10-06T14:30:45Z",
    "account_number": "+44-7700-900456",
    "account_age_days": 89,
    "current_location": "Mumbai, India",
    "request_origin_location": "Lagos, Nigeria",
    "last_activity_location": "Mumbai, India",
    "last_activity_time": "2025-10-06T12:15:20Z",
    "customer_verified": False,
    "sms_verification_sent": True,
    "sms_verification_response": None,
    "recent_password_change": True,
    "password_changed_time": "2025-10-06T13:45:10Z",
    "similar_fraud_pattern": "FR-2847",
    "account_value_gbp": 850,
    "payment_history": "good",
    "previous_sim_swaps": 0,
    "customer_support_contacts_30d": 0
}

print("Sample SIM Swap Request Data:")
print(json.dumps(SIM_SWAP_REQUEST, indent=2))

Sample SIM Swap Request Data:
{
  "request_id": "SIM-2025-10-06-48271",
  "timestamp": "2025-10-06T14:30:45Z",
  "account_number": "+44-7700-900456",
  "account_age_days": 89,
  "current_location": "Mumbai, India",
  "request_origin_location": "Lagos, Nigeria",
  "last_activity_location": "Mumbai, India",
  "last_activity_time": "2025-10-06T12:15:20Z",
  "customer_verified": false,
  "sms_verification_sent": true,
  "sms_verification_response": null,
  "recent_password_change": true,
  "password_changed_time": "2025-10-06T13:45:10Z",
  "similar_fraud_pattern": "FR-2847",
  "account_value_gbp": 850,
  "payment_history": "good",
  "previous_sim_swaps": 0,
  "customer_support_contacts_30d": 0
}


### Step 2: TODO - Design Your Prompt

**🎯 Your Turn!**

In the cell below, design a prompt for the SIM Swap Investigation Agent. Consider:

1. **Role**: What expertise should the AI have?
2. **Task**: What exactly should it analyze?
3. **Constraints**: What should it focus on? What should it avoid?
4. **Output**: What structure should the report have?

Think about:
- What are red flags for SIM swap fraud?
- What makes a request legitimate?
- What actions should be recommended?
- How confident should the AI be before recommending blocking?

### Step 3: Implement Your Agent

In [ ]:
# TODO: Implement your SIM Swap Investigation Agent

class SIMSwapInvestigationAgent(BaseAgent):
    """Agent that investigates SIM swap requests for potential fraud."""

    def __init__(self, llm_client):
        # TODO: Design your system prompt here
        # Follow the Role + Task + Constraints + Output framework

        system_prompt = """You are an expert telecom fraud investigator with 15 years of experience analyzing SIM swap fraud patterns.

TASK: Investigate this SIM swap request and provide a detailed analysis with recommendation.

AVAILABLE CONTEXT:
- Fraud pattern FR-2847: Known fraud ring operating from West Africa targeting new accounts with sudden geographic anomalies
- Industry standard: SIM swap from different country than last activity = 78% fraud rate
- Company policy: Block if confidence > 85%, Escalate if 60-85%, Approve if < 60%

PROVIDE INVESTIGATION REPORT IN THIS STRUCTURE (in Persian):

1. RECOMMENDATION: [APPROVE/BLOCK/ESCALATE]
2. CONFIDENCE LEVEL: [percentage and reasoning]

3. RISK FACTORS IDENTIFIED:
   - [List each red flag with severity: HIGH/MEDIUM/LOW]

4. PROTECTIVE FACTORS:
   - [List any factors suggesting legitimate request]

5. GEOGRAPHIC ANALYSIS:
   - [Evaluate location consistency and travel possibility]

6. TEMPORAL ANALYSIS:
   - [Evaluate timing of events and sequences]

7. PATTERN MATCHING:
   - [Compare to known fraud patterns]

8. CUSTOMER BEHAVIOR:
   - [Evaluate response to verification, history]

9. DETAILED REASONING:
   - [Explain your analytical process]

10. RECOMMENDED ACTIONS:
    - [Specific steps for fraud team]

11. FALSE POSITIVE RISK:
    - [Estimate likelihood this is legitimate]

Keep analysis thorough but concise. Focus on actionable intelligence."""

        super().__init__(llm_client, system_prompt)

    async def process(self, input: IAgentInput) -> Dict[str, Any]:
        """Investigate SIM swap request."""

        # Format the request data nicely
        request_data = input.metadata.get('request_data', {})
        formatted_request = json.dumps(request_data, indent=2)

        llm_input = ILLMInput(
            system_prompt=self.system_prompt,
            user_message=f"SIM Swap Request Data:\n{formatted_request}"
        )

        result = await self.llm_client.chat(llm_input)

        return {
            "investigation_report": result.get('llm_response', ''),
            "request_id": request_data.get('request_id', 'unknown'),
            "investigated_at": datetime.now().isoformat()
        }

print("✅ SIMSwapInvestigationAgent created")

✅ SIMSwapInvestigationAgent created


### Step 4: Test Your Agent

In [ ]:
# Create test dataset with various scenarios
SIM_SWAP_TEST_CASES = [
    {
        "name": "Highly Suspicious - Geographic Impossibility",
        "data": SIM_SWAP_REQUEST  # The one we defined earlier
    },
    {
        "name": "Legitimate - Traveling Customer",
        "data": {
            "request_id": "SIM-2025-10-06-48352",
            "timestamp": "2025-10-06T10:15:30Z",
            "account_number": "+44-7700-900789",
            "account_age_days": 1247,
            "current_location": "Dubai, UAE",
            "request_origin_location": "Dubai, UAE",
            "last_activity_location": "London, UK",
            "last_activity_time": "2025-10-05T08:30:00Z",
            "customer_verified": True,
            "sms_verification_sent": True,
            "sms_verification_response": "verified_within_2_minutes",
            "recent_password_change": False,
            "password_changed_time": None,
            "similar_fraud_pattern": None,
            "account_value_gbp": 1850,
            "payment_history": "excellent",
            "previous_sim_swaps": 1,
            "customer_support_contacts_30d": 1,
            "support_contact_reason": "Informed about travel to Dubai, requested international roaming"
        }
    },
    {
        "name": "Suspicious - Multiple Red Flags but Lower Confidence",
        "data": {
            "request_id": "SIM-2025-10-06-48423",
            "timestamp": "2025-10-06T16:45:20Z",
            "account_number": "+44-7700-900321",
            "account_age_days": 156,
            "current_location": "Manchester, UK",
            "request_origin_location": "Birmingham, UK",
            "last_activity_location": "Manchester, UK",
            "last_activity_time": "2025-10-06T14:20:10Z",
            "customer_verified": False,
            "sms_verification_sent": True,
            "sms_verification_response": None,
            "recent_password_change": True,
            "password_changed_time": "2025-10-06T16:00:00Z",
            "similar_fraud_pattern": None,
            "account_value_gbp": 450,
            "payment_history": "fair",
            "previous_sim_swaps": 0,
            "customer_support_contacts_30d": 0
        }
    }
]

# Create agent
sim_swap_agent = SIMSwapInvestigationAgent(llm_client)

# Test with all cases
print("\n" + "="*80)
print("SIM SWAP INVESTIGATION RESULTS")
print("="*80)

for test_case in SIM_SWAP_TEST_CASES:
    print(f"\n📱 Test Case: {test_case['name']}")
    print("-" * 80)

    agent_input = IAgentInput(
        message="Investigate SIM swap request",
        metadata={"request_data": test_case['data']}
    )

    result = await sim_swap_agent.process(agent_input)

    print(result['investigation_report'])
    print(f"\n⏱️  Investigation time: <8 seconds (vs 5-10 minutes manual)")
    print("-" * 80)

print("\n" + "="*80)
print("✅ All investigations complete!")


SIM SWAP INVESTIGATION RESULTS

📱 Test Case: Highly Suspicious - Geographic Impossibility
--------------------------------------------------------------------------------
2025-10-06 19:44:13.494 | INFO     | arshai.OpenRouterClient:callHandlers:1762 - Processing chat request - Regular Functions: False, Background: False, Structured: False
2025-10-06 19:44:14.337 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-06 19:44:14.339 | INFO     | arshai.httpx:callHandlers:1762 - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
**گزارش تحقیق در مورد درخواست تعویض سیم کارت**

1. **توصیه:** BLOCK

2. **سطح اطمینان:** 90% - با توجه به الگوی جغرافیایی غیرمعمول و تطابق با الگوی تقلب شناخته شده FR-2847، سطح اطمینان بالاست.

3. **عوامل ریسک شناسایی شده:**
   - **مکان درخواست:** درخواست از لاگوس، نیجریه در حالی که آخرین فعالیت در بمبئی، هند بوده است. (HIGH)
   - **عدم تأیید مشتری:*

### Step 5: Calculate ROI

In [ ]:
# ROI Calculation
print("\n" + "="*80)
print("ROI CALCULATION: SIM SWAP INVESTIGATION AUTOMATION")
print("="*80)

# Parameters
daily_flagged_requests = 1500
manual_time_per_request_minutes = 7.5  # Average of 5-10 minutes
ai_time_per_request_seconds = 8
analyst_cost_per_hour = 50  # GBP
human_review_time_minutes = 0.75  # 45 seconds to review AI report

# Manual process
manual_hours_per_day = (daily_flagged_requests * manual_time_per_request_minutes) / 60
manual_cost_per_day = manual_hours_per_day * analyst_cost_per_hour
manual_requests_reviewed = 200  # Capacity limit
manual_uninvestigated = daily_flagged_requests - manual_requests_reviewed

# AI process
ai_processing_hours = (daily_flagged_requests * ai_time_per_request_seconds) / 3600
human_review_hours = (daily_flagged_requests * human_review_time_minutes) / 60
total_ai_process_hours = ai_processing_hours + human_review_hours
ai_cost_per_day = human_review_hours * analyst_cost_per_hour  # AI processing cost negligible
ai_requests_reviewed = daily_flagged_requests  # Can handle all

# Savings
time_savings_hours = manual_hours_per_day - total_ai_process_hours
cost_savings_per_day = manual_cost_per_day - ai_cost_per_day
cost_savings_per_month = cost_savings_per_day * 30
cost_savings_per_year = cost_savings_per_day * 365
time_reduction_percent = (time_savings_hours / manual_hours_per_day) * 100

print(f"\n📊 MANUAL PROCESS:")
print(f"   Daily flagged requests: {daily_flagged_requests:,}")
print(f"   Time per investigation: {manual_time_per_request_minutes} minutes")
print(f"   Total time required: {manual_hours_per_day:.1f} hours/day")
print(f"   Requests actually reviewed: {manual_requests_reviewed} ({manual_requests_reviewed/daily_flagged_requests*100:.1f}%)")
print(f"   Uninvestigated requests: {manual_uninvestigated} per day")
print(f"   Daily cost: £{manual_cost_per_day:,.2f}")

print(f"\n🤖 AI-POWERED PROCESS:")
print(f"   Daily flagged requests: {daily_flagged_requests:,}")
print(f"   AI processing time: {ai_time_per_request_seconds} seconds per request")
print(f"   Human review time: {human_review_time_minutes*60:.0f} seconds per request")
print(f"   Total time required: {total_ai_process_hours:.1f} hours/day")
print(f"   Requests reviewed: {ai_requests_reviewed} (100%)")
print(f"   Uninvestigated requests: 0")
print(f"   Daily cost: £{ai_cost_per_day:,.2f}")

print(f"\n💰 SAVINGS:")
print(f"   Time saved per day: {time_savings_hours:.1f} hours ({time_reduction_percent:.1f}% reduction)")
print(f"   Cost saved per day: £{cost_savings_per_day:,.2f}")
print(f"   Cost saved per month: £{cost_savings_per_month:,.2f}")
print(f"   Cost saved per year: £{cost_savings_per_year:,.2f}")
print(f"\n📈 ADDITIONAL BENEFITS:")
print(f"   Additional requests investigated: {ai_requests_reviewed - manual_requests_reviewed:,} per day")
print(f"   Coverage improvement: {(ai_requests_reviewed/daily_flagged_requests - manual_requests_reviewed/daily_flagged_requests)*100:.1f}%")
print(f"   Consistency: 100% (no analyst fatigue)")
print(f"   Availability: 24/7")

print("\n" + "="*80)


ROI CALCULATION: SIM SWAP INVESTIGATION AUTOMATION

📊 MANUAL PROCESS:
   Daily flagged requests: 1,500
   Time per investigation: 7.5 minutes
   Total time required: 187.5 hours/day
   Requests actually reviewed: 200 (13.3%)
   Uninvestigated requests: 1300 per day
   Daily cost: £9,375.00

🤖 AI-POWERED PROCESS:
   Daily flagged requests: 1,500
   AI processing time: 8 seconds per request
   Human review time: 45 seconds per request
   Total time required: 22.1 hours/day
   Requests reviewed: 1500 (100%)
   Uninvestigated requests: 0
   Daily cost: £937.50

💰 SAVINGS:
   Time saved per day: 165.4 hours (88.2% reduction)
   Cost saved per day: £8,437.50
   Cost saved per month: £253,125.00
   Cost saved per year: £3,079,687.50

📈 ADDITIONAL BENEFITS:
   Additional requests investigated: 1,300 per day
   Coverage improvement: 86.7%
   Consistency: 100% (no analyst fatigue)
   Availability: 24/7



---

## Summary: Day 1 Learnings

### What We Covered

1. **Prompt Engineering Framework**
   - Role + Task + Constraints + Output
   - Bad vs. good prompt comparison
   - Importance of structure and specificity

2. **Single LLM Call Use Cases**
   - Email Threat Detection
   - Incident Report Generation
   - Network Anomaly Explanation
   - CDR Fraud Analysis
   - Log Contextualization
   - SIM Swap Investigation (hands-on)

3. **Arshai Framework Basics**
   - Direct instantiation pattern
   - Creating custom agents
   - ILLMInput and IAgentInput interfaces
   - Async processing

### Key Takeaways

✅ **Single LLM calls can solve 80% of repetitive security tasks**

✅ **Good prompts = good outputs** - Invest time in prompt engineering

✅ **ROI is immediate and measurable** - 88-99% time savings

✅ **AI doesn't replace analysts** - It frees them for high-value work

✅ **Consistency matters** - AI provides uniform analysis 24/7

### What's Next: Day 2

Tomorrow we'll explore:
- **Multi-agent systems** - Coordinating multiple AI agents
- **Tool calling** - AI agents that use external tools and APIs
- **Alert correlation** - Connecting dots across multiple systems
- **Autonomous threat hunting** - AI that proactively searches for threats
- **Autonomous response** - When should AI take action automatically?

---

## Exercises to Try on Your Own

1. **Modify prompts**: Try adjusting the temperature, max_tokens, or prompt wording to see how it affects output

2. **Create your own agent**: Think of a repetitive security task in your organization and create an agent for it

3. **Test edge cases**: What happens with malformed input? How does the agent handle ambiguous data?

4. **Calculate your ROI**: Use real numbers from your organization to calculate potential savings

5. **Expand the SIM Swap agent**: Add more sophisticated analysis, confidence scoring, or integration with external data sources

---

## Resources

- **Arshai Documentation**: https://github.com/felesh-ai/arshai
- **OpenRouter Models**: https://openrouter.ai/models
- **Prompt Engineering Guide**: https://www.promptingguide.ai/
- **Telecom Fraud**: CFCA Fraud Loss Survey

---

### Questions?

Feel free to experiment with the code, modify the agents, and test with your own data!

**See you tomorrow for Day 2! 🚀**